# Session 08: Async Debugging Lab

**Notebook 3 of 3: Debugging lab** · used in the live session's debugging segment, and for practice afterwards (about 45 minutes)

Six small async programs, each broken in a way that's common in real code. Most of them **don't crash**: they're slow, or quietly wrong, which is what makes async bugs expensive. For each one:

1. **Observe:** run the broken cell and read the output.
2. **Hypothesize:** write down what you think is wrong *before* opening a hint.
3. **Inspect:** use the hints if you need them.
4. **Fix:** edit the code in the fix cell.
5. **Verify:** run the fix cell; its check prints `PASS` or `FAIL`.

Everything runs in this kernel, which already has an event loop, so cells can `await` directly. Run the next cell first.

In [ ]:
import asyncio
import time
import warnings

import httpx

warnings.simplefilter("always", RuntimeWarning)  # show "never awaited" warnings every time


def check(condition: bool, message: str) -> None:
    print(("PASS  " if condition else "FAIL  ") + message)


async def slow_lookup(team: str) -> list[str]:
    """Stands in for a network call that takes 0.2 s."""
    await asyncio.sleep(0.2)
    return [f"{team}-asset"]


print("ready")

## Lab 1: The list that isn't a list

A helper counts a team's assets. It crashes, and the error message mentions a type you didn't expect.

In [ ]:
async def count_assets(team: str) -> int:
    assets = slow_lookup(team)
    return len(assets)


try:
    print(await count_assets("network"))
except TypeError as error:
    print("TypeError:", error)

**Observe:** what type does `len()` complain about? Look for a `RuntimeWarning` in the output too.

<details>
<summary>Hint 1</summary>

**Inspect** with `print(type(assets))` straight after the call. What does calling an `async def` function return?

</details>

<details>
<summary>Hint 2: the explanation</summary>

`slow_lookup(team)` returns a **coroutine object**: the work hasn't run. It needs `await`: `assets = await slow_lookup(team)`. The warning "coroutine 'slow_lookup' was never awaited" is Python telling you a coroutine was thrown away unrun.

In JavaScript or C#, forgetting `await` still runs the work; in Python, it **doesn't**. If you see "was never awaited" in your logs, some work silently didn't happen.

</details>

In [ ]:
# Your fix
async def count_assets(team: str) -> int:
    assets = slow_lookup(team)
    return len(assets)


try:
    check(await count_assets("network") == 1, "count_assets returns 1 for one asset")
except TypeError as error:
    check(False, f"still a TypeError: {error}")

## Lab 2: Concurrent, but not faster

A teammate "made the lookup async" and used `gather`. The dashboard is still exactly as slow as before.

In [ ]:
async def legacy_lookup(team: str) -> list[str]:
    time.sleep(0.2)  # the old synchronous client's call
    return [f"{team}-asset"]


started = time.perf_counter()
await asyncio.gather(
    *(legacy_lookup(team) for team in ["database", "it-support", "network", "platform"])
)
print(f"four lookups: {time.perf_counter() - started:.2f}s")

**Observe:** four lookups of 0.2 s each. If they overlapped, how long should it take?

<details>
<summary>Hint 1</summary>

**Inspect:** where does `legacy_lookup` give control back to the event loop? Find its `await`s.

</details>

<details>
<summary>Hint 2: the explanation</summary>

There are none. `time.sleep` (like `requests.get` or `httpx.Client.get`) **blocks the thread**, and the event loop runs on that thread, so each coroutine runs start to finish before the next one can start. Marking a function `async def` doesn't make its body asynchronous.

**Fix:** await something that yields while waiting: `await asyncio.sleep(0.2)` here, `await client.get(...)` with an `httpx.AsyncClient` in real code. If a library has no async version, `await asyncio.to_thread(blocking_call, ...)` runs it in a thread.

In a FastAPI `async def` endpoint, this bug also stalls **every other request** to the service.

</details>

In [ ]:
# Your fix
async def legacy_lookup(team: str) -> list[str]:
    time.sleep(0.2)
    return [f"{team}-asset"]


started = time.perf_counter()
await asyncio.gather(
    *(legacy_lookup(team) for team in ["database", "it-support", "network", "platform"])
)
elapsed = time.perf_counter() - started
check(elapsed < 0.5, f"four lookups overlap (took {elapsed:.2f}s)")

## Lab 3: One failure, no dashboard

The inventory is down for one team. The dashboard should show the other three teams, and say which one is missing. Instead, it shows nothing.

In [ ]:
async def team_lookup(team: str) -> list[str]:
    await asyncio.sleep(0.1)
    if team == "network":
        raise ConnectionError("inventory unavailable for network")
    return [f"{team}-asset"]


async def dashboard(teams: list[str]) -> dict[str, object]:
    results = await asyncio.gather(*(team_lookup(team) for team in teams))
    return dict(zip(teams, results))


try:
    print(await dashboard(["database", "it-support", "network", "platform"]))
except ConnectionError as error:
    print("dashboard failed:", error)

**Observe:** three lookups succeeded. Where did their results go?

<details>
<summary>Hint 1</summary>

**Inspect** what `gather` does when one of its coroutines raises. Is that the behaviour this dashboard needs?

</details>

<details>
<summary>Hint 2: the explanation</summary>

By default, `gather` raises the **first** exception, and the other results are lost to the caller (the other coroutines even keep running). That's right when the results only make sense together, and wrong for a dashboard, where a partial answer is useful.

**Fix**, two options:

- Handle the expected failure **per team**: wrap the call in a small coroutine that catches `ConnectionError` and returns, say, `{"error": str(error)}`. Unexpected exceptions still propagate. This is the workbook's pattern.
- Or `gather(..., return_exceptions=True)`, then check each result with `isinstance(result, Exception)`.

The check expects the network team's value to be a dict with an `"error"` key.

</details>

In [ ]:
# Your fix
async def dashboard(teams: list[str]) -> dict[str, object]:
    results = await asyncio.gather(*(team_lookup(team) for team in teams))
    return dict(zip(teams, results))


try:
    board = await dashboard(["database", "it-support", "network", "platform"])
    check(board["database"] == ["database-asset"], "other teams still have their assets")
    check(
        isinstance(board["network"], dict) and "error" in board["network"], "network shows an error"
    )
except ConnectionError as error:
    check(False, f"the whole dashboard still fails: {error}")

## Lab 4: Works as a script, fails in the notebook

This code was copied from a working script. In the notebook (and inside a FastAPI `async def` endpoint) it fails.

In [ ]:
async def main() -> list[str]:
    return await slow_lookup("platform")


try:
    print(asyncio.run(main()))
except RuntimeError as error:
    print("RuntimeError:", error)

**Observe:** read the error message closely.

<details>
<summary>Hint: the explanation</summary>

`asyncio.run()` **creates** an event loop, runs a coroutine on it, and closes it. It's the entry point of an async **program**, called once from synchronous code: `if __name__ == "__main__": asyncio.run(main())`.

Jupyter, FastAPI and pytest-asyncio already run an event loop, and you can't start a second one on the same thread. Inside them, you're already in async code, so just `await main()`.

The same error appears when a sync helper deep inside an async application calls `asyncio.run()`. The fix there is to make the helper `async def` and await it all the way up.

(Search results sometimes suggest the `nest_asyncio` package. It patches asyncio to allow nested loops; it hides the design problem rather than fixing it.)

</details>

In [ ]:
# Your fix
async def main() -> list[str]:
    return await slow_lookup("platform")


try:
    result = asyncio.run(main())
    check(result == ["platform-asset"], "main() ran in the notebook")
except RuntimeError as error:
    check(False, f"still: {error}")

## Lab 5: The deadline that doesn't stop anything

A report step has a 0.3-second deadline. It takes a full second, and the deadline never fires.

In [ ]:
def crunch_numbers() -> int:
    total = 0
    deadline_end = time.perf_counter() + 1.0  # about one second of pure CPU work
    while time.perf_counter() < deadline_end:
        total += 1
    return total


async def build_report() -> str:
    crunch_numbers()
    return "report"


started = time.perf_counter()
try:
    async with asyncio.timeout(0.3):
        await build_report()
    print(f"no TimeoutError, and it took {time.perf_counter() - started:.2f}s")
except TimeoutError:
    print(f"TimeoutError after {time.perf_counter() - started:.2f}s")

**Observe:** the deadline was 0.3 s. Why was there no `TimeoutError`?

<details>
<summary>Hint 1</summary>

**Inspect:** how does `asyncio.timeout` stop the code inside it? Where can that happen?

</details>

<details>
<summary>Hint 2: the explanation</summary>

Cancellation is **cooperative**: the timeout cancels the coroutine by raising `CancelledError` at an `await`. `build_report` never awaits anything that waits, so the event loop doesn't get control back until the work is done, and by then there's nothing left to cancel. (A blocking `time.sleep` or synchronous HTTP call behaves the same way.) A deadline you can't enforce is worse than none, because it looks like protection.

**Fix:** run the blocking work in a thread, so the event loop is free to enforce the deadline: `await asyncio.to_thread(crunch_numbers)`. The deadline then fires on time. Note that the thread itself keeps running in the background until it finishes: threads can't be cancelled from outside, so a real fix for long CPU work also needs a way to stop early, or a separate process.

</details>

In [ ]:
# Your fix
async def build_report() -> str:
    crunch_numbers()
    return "report"


started = time.perf_counter()
try:
    async with asyncio.timeout(0.3):
        await build_report()
    check(False, "the deadline should have fired")
except TimeoutError:
    elapsed = time.perf_counter() - started
    check(elapsed < 0.6, f"the deadline fired on time ({elapsed:.2f}s)")
await asyncio.sleep(1.0)  # let any background thread from your fix finish before the next lab

## Lab 6: Lost updates, with no threads in sight

The service counts how many inventory calls it makes, for its metrics page. After 50 concurrent calls, the counter says far fewer. "But async is single-threaded, so there are no race conditions!"

In [ ]:
calls_made = 0


async def counted_lookup(team: str) -> list[str]:
    global calls_made
    current = calls_made
    assets = await slow_lookup(team)  # other coroutines run while this one waits
    calls_made = current + 1
    return assets


await asyncio.gather(*(counted_lookup(f"team-{n}") for n in range(50)))
print("calls made:", calls_made)

**Observe:** 50 calls were made. What does the counter say, and why exactly that number?

<details>
<summary>Hint 1</summary>

**Inspect:** what value of `calls_made` does each coroutine read, and when does it write it back? What happens between the two?

</details>

<details>
<summary>Hint 2: the explanation</summary>

All 50 coroutines read `calls_made` (0) **before** any of them writes, because each then waits at `await`, and the loop runs the next one. After the wait, each writes `0 + 1`. The result is `1`.

Coroutines can't be interrupted **between** `await`s, but anything can happen **at** an `await`. A read-modify-write that spans an `await` is a race, just as with threads.

**Fix:** don't hold state across the `await`: increment in one step, `calls_made += 1`, before or after the call (no `await` in between). If a longer critical section truly must span an `await`, protect it with an `asyncio.Lock`.

</details>

In [ ]:
# Your fix
calls_made = 0


async def counted_lookup(team: str) -> list[str]:
    global calls_made
    current = calls_made
    assets = await slow_lookup(team)
    calls_made = current + 1
    return assets


await asyncio.gather(*(counted_lookup(f"team-{n}") for n in range(50)))
check(calls_made == 50, f"counter says {calls_made}")

## Wrap-up

Which of these six would your tests have caught?

- Labs 1 and 4 crash, so any test that runs the code catches them.
- Labs 2 and 5 are **performance** bugs: correct results, wrong timing. They need a test that checks concurrency (a counter of requests in flight, as in the workbook's `FakeInventory`) or a deadline.
- Labs 3 and 6 are **partial-failure** and **interleaving** bugs: they appear only when several things happen at once, so tests must run things concurrently, with failures injected.

Write down, in your workbook notes, which of these mistakes you think you're most likely to make, and how you'd notice it in production.